In [ ]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = os.getcwd() if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

In [ ]:
%%capture

# Compile Cython files
os.chdir(LOCAL_REPO_PATH)
!python Challenge/compile_cython.py . --inplace
os.chdir(WORKING_DIR)

In [ ]:
%%capture

!pip install optuna
import optuna

In [ ]:
import importlib
import scipy.sparse as sps
import pandas as pd
import numpy as np

from Challenge import paths
importlib.reload(paths)

from Challenge.hyper_tuning import hyperparameter_tuning

In [ ]:
# Load datasets
URM_train = sps.load_npz(paths.URM_TRAIN)
URM_validation = sps.load_npz(paths.URM_VALIDATION)

In [ ]:
def evaluate_recommender(recommender, at):
    cumulative_recall = 0.0
    num_eval = 0
    
    for user_id in range(URM_validation.shape[0]):
        relevant_items = URM_validation.indices[URM_validation.indptr[user_id]:URM_validation.indptr[user_id+1]]
        
        if len(relevant_items)>0:
            num_eval+=1
            
            recommended_items = recommender.recommend(user_id, cutoff=at)
            
            is_relevant = np.isin(recommended_items, relevant_items, assume_unique=True)
            recall_score = np.sum(is_relevant, dtype=np.float32) / relevant_items.shape[0]

            cumulative_recall += recall_score

    return cumulative_recall / num_eval

In [ ]:
from Recommenders.MatrixFactorization.IALSRecommender import IALSRecommender

# Load IALS model
ials_model = IALSRecommender.load_model(paths.MODEL_DIR)

# Evaluate IALS model
ials_recall = evaluate_recommender(ials_model, at=20)
print(f"IALS Model - Recall@20: {ials_recall:.5f}")

In [ ]:
import optuna
from Recommenders.SLIM.SLIMElasticNetRecommender import SLIMElasticNetRecommender

# Load SLIM optuna_study to get best hyperparameters
slim_study = optuna.load_study(study_name=SLIMElasticNetRecommender.RECOMMENDER_NAME+"_refined")

# Train SLIM model with best hyperparameters
slim_model = SLIMElasticNetRecommender(URM_train)
best_slim_params = slim_study.best_params
slim_model.fit(**best_slim_params)

# Evaluate SLIM model
slim_recall = evaluate_recommender(slim_model, at=20)
print(f"SLIM Model - Recall@20: {slim_recall:.5f}")

In [ ]:
from Recommenders.KNN.ItemKNNCBFRecommender import ItemKNNCBFRecommender

STUDY_NAME = ItemKNNCBFRecommender.RECOMMENDER_NAME + "_refined"

# Must be retrained because the model was not saved
def objective_function(trial):    
    recommender_instance = ItemKNNCBFRecommender(URM_train)
    recommender_instance.fit(
        topK=trial.suggest_int("topK", 5, 10000),
        shrink=trial.suggest_int("shrink", 0, 8000),
        similarity="cosine",
        normalize=True
    )

    return evaluate_recommender(recommender_instance, at=20)

In [ ]:
knn_study = hyperparameter_tuning(
    study_name=STUDY_NAME,
    objective_function=objective_function,
    n_trials=100
)

In [ ]:
knn_model = ItemKNNCBFRecommender(URM_train)
knn_model.fit(
    topK=knn_study.best_trial.params["topK"],
    shrink=knn_study.best_trial.params["shrink"],
    similarity="cosine",
    feature_weighting="TF-IDF",
    normalize=True
)

# Evaluate KNN model
knn_recall = evaluate_recommender(knn_model, at=20)
print(f"KNN Model - Recall@20: {knn_recall:.5f}")

In [ ]:
from typing import List
from Recommenders.BaseRecommender import BaseRecommender

class HybridRecommender():
    RECOMMENDER_NAME = "HybridRecommender"
    
    def __init__(self, recommenders: List[BaseRecommender]):
        self.recommenders = recommenders

        self.set_hyperparameters()

    def set_hyperparameters(self, weights: List[float]=None, cutoff_multiplier: float=1.0):
        if weights is None:
            self.weights = [1.0 / len(self.recommenders)] * len(self.recommenders)
        else:
            self.weights = weights

        self.cutoff_multiplier = cutoff_multiplier

    def recommend(self, user_id, cutoff=20):
        cutoff_adjusted = int(cutoff * self.cutoff_multiplier)

        item_scores = {}
        for idx, recommender in enumerate(self.recommenders):
            ranking_list, scores_batch = recommender.recommend(user_id, cutoff=cutoff_adjusted, return_scores=True)

            # Normalize scores
            rec_scores = scores_batch
            rec_scores = (rec_scores - np.min(rec_scores)) / (np.max(rec_scores) - np.min(rec_scores) + 1e-8)
            
            for item_id, score in zip(ranking_list, rec_scores):
                score *= self.weights[idx]

                if item_id in item_scores:
                    item_scores[item_id] += score
                else:
                    item_scores[item_id] = score

        # Sort scores
        scores = np.array(list(item_scores.values()))

        # Get top N recommendations
        top_items = np.argsort(-scores)[:cutoff]
        
        return top_items
    
    def save_model(self, folder_path, file_name="hybrid"):
        for idx, recommender in enumerate(self.recommenders):
            recommender.save_model(folder_path, file_name + f"_submodel_{idx}")

    def load_model(self, recommender_classes, file_name="hybrid"):
        self.recommenders = []
        for idx, recommender_class in enumerate(recommender_classes):
            recommender = recommender_class.load_model(paths.MODEL_DIR, file_name + f"_submodel_{idx}")
            self.recommenders.append(recommender)

In [ ]:
from optuna.exceptions import TrialPruned

STUDY_NAME = HybridRecommender.RECOMMENDER_NAME

# Must be retrained because the model was not saved
def hybrid_objective(trial):
    hybrid_model = HybridRecommender(
        recommenders=[ials_model, slim_model, knn_model]
    )
    
    cutoff_multiplier = trial.suggest_float("cutoff_multiplier", 1, 5.0)
    ials_weight = trial.suggest_float("ials_weight", 0.0, 1.0)
    slim_weight = trial.suggest_float("slim_weight", 0.0, 1.0)
    
    # Ensure weights sum to 1
    if ials_weight + slim_weight > 1.0:
        raise TrialPruned()
    
    knn_weight = 1.0 - ials_weight - slim_weight

    hybrid_model.set_hyperparameters(
        weights=[
            ials_weight,
            slim_weight,
            knn_weight
        ],
        cutoff_multiplier=cutoff_multiplier
    )

    return evaluate_recommender(hybrid_model, at=20)

In [ ]:
# Perform hyperparameter tuning
save_results, optuna_study = hyperparameter_tuning(
    hybrid_objective,
    study_name=STUDY_NAME,
    n_trials=100
)

In [ ]:
optuna.visualization.plot_optimization_history(optuna_study)

In [ ]:
optuna.visualization.plot_param_importances(optuna_study)

In [ ]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

In [ ]:
# Load the best SLIM model, IALS model is trained looking at validation already
slim_best = SLIMElasticNetRecommender(URM_train+URM_validation).load_model(paths.MODEL_DIR)

# Train ItemKNNCF model
knn_best = ItemKNNCBFRecommender(URM_train+URM_validation)
knn_best.fit(
    topK=knn_study.best_trial.params["topK"],
    shrink=knn_study.best_trial.params["shrink"],
    similarity="cosine",
    feature_weighting="TF-IDF",
    normalize=True
)

In [ ]:
# Initialize the best model
best_param = optuna_study.best_trial.params
hybrid_model = HybridRecommender(
    recommenders=[ials_model, slim_model, knn_model]
)
hybrid_model.set_hyperparameters(
    weights=[
        best_param["ials_weight"],
        best_param["slim_weight"],
        1.0 - best_param["ials_weight"] - best_param["slim_weight"]
    ],
    cutoff_multiplier=best_param["cutoff_multiplier"]
)

In [ ]:
# Print performance of each individual model
ials_recall = evaluate_recommender(ials_model, at=20)
print(f"IALS Model - Recall@20: {ials_recall:.5f}")
slim_recall = evaluate_recommender(slim_best, at=20)
print(f"SLIM Model - Recall@20: {slim_recall:.5f}")
knn_recall = evaluate_recommender(knn_best, at=20)
print(f"KNN Model - Recall@20: {knn_recall:.5f}")
# Print performance of hybrid model (Biased because trained on train+validation)
hybrid_recall = evaluate_recommender(hybrid_model, at=20)
print(f"Hybrid Model - Recall@20: {hybrid_recall:.5f}")

In [ ]:
import pandas as pd

# Generate recommendations for the test set
user_ids_test = pd.read_csv(paths.CHALLENGE_USER_IDS_TEST)
ids = user_ids_test["user_id"].values

recommendations = hybrid_model.recommend(ids, cutoff=20)

os.makedirs(paths.SUBMISSIONS, exist_ok=True)
with open(os.path.join(paths.SUBMISSIONS, STUDY_NAME + ".csv"), "w") as f:
    f.write("user_id,item_list\n")
    for user_id, rec_list in zip(ids, recommendations):
        f.write(f"{user_id},{' '.join([str(item) for item in rec_list])}\n")